# ניתוח אינטגרציה: trips (מתוכנן) מול gtfs_rides (בפועל)

מטרה: לבדוק האם 48 הקווים מ-`trips.csv` (כולם מסומנים `low_demand_flag=True`) מופיעים בנתוני הנסיעות בפועל (`gtfs_rides`), באיזו תדירות פעלו ב-april/may, והאם זמני ההתחלה תואמים.

**מפתח חיבור:** `trips.catalog_number`  ↔  `gtfs_rides.gtfs_route__route_mkt` (מק"ט הקו).
ב-`trips` שדה `sign` הוא מספר הקו (route_short_name) ו-`catalog_number` הוא המק"ט.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')

# מתוכנן
trips = pd.read_csv(DATA_DIR / 'trips.csv')

# בפועל — אפריל ושני ה-batches של מאי
apr = pd.read_csv(DATA_DIR / 'gtfs_rides_apr.csv')
may = pd.concat([
    pd.read_csv(DATA_DIR / 'gtfs_rides_may_b1.csv'),
    pd.read_csv(DATA_DIR / 'gtfs_rides_may_b2.csv'),
], ignore_index=True)

# נירמול טיפוסים + זמן מקומי (הנתונים ב-UTC)
for df in (apr, may):
    df['mkt'] = df['gtfs_route__route_mkt'].astype(str)
    df['line'] = df['gtfs_route__route_short_name'].astype(str)
    df['start_time'] = pd.to_datetime(df['start_time'], utc=True)
    df['start_local'] = df['start_time'].dt.tz_convert('Asia/Jerusalem')
    df['date'] = df['gtfs_route__date']
    df['hour'] = df['start_local'].dt.hour

print('trips:', trips.shape, '| apr rides:', apr.shape, '| may rides:', may.shape)
print('apr days:', apr['date'].nunique(), '| may days:', may['date'].nunique())

trips: (1421, 22) | apr rides: (5489, 21) | may rides: (4410, 21)
apr days: 5 | may days: 4


## שאלה 1 — האם הקווים מ-trips מופיעים ב-rides?
בדיקה לפי מק"ט (`catalog_number` ↔ `route_mkt`), ובדיקה משנית לפי מספר הקו (`sign` ↔ `route_short_name`).

In [2]:
trips_mkt = set(trips['catalog_number'].astype(str))
rides_mkt = set(apr['mkt']) | set(may['mkt'])

overlap_mkt = trips_mkt & rides_mkt
print('מק"טים ב-trips :', len(trips_mkt))
print('מק"טים ב-rides :', len(rides_mkt))
print('חפיפה לפי מק"ט:', len(overlap_mkt), '->', sorted(overlap_mkt))
print()

# בדיקה משנית לפי מספר הקו (sign)
trips_sign = set(trips['sign'].astype(str))
rides_line = set(apr['line']) | set(may['line'])
overlap_sign = trips_sign & rides_line
print('חפיפה לפי מספר-קו (sign↔short_name):', sorted(overlap_sign))
print('  (קווים 6/8 הם מספרים נפוצים ארצית — בעלי מק"ט שונה — ולכן ההתאמה כאן מקרית)')

מק"טים ב-trips : 48
מק"טים ב-rides : 17
חפיפה לפי מק"ט: 0 -> []

חפיפה לפי מספר-קו (sign↔short_name): ['6', '8']
  (קווים 6/8 הם מספרים נפוצים ארצית — בעלי מק"ט שונה — ולכן ההתאמה כאן מקרית)


**ממצא:** אין ולו מק"ט אחד משותף. 48 הקווים של `trips` ו-17 הקווים שב-`rides` הם שתי קבוצות **זרות לחלוטין**.
מכאן שלא ניתן לבצע חיבור מתוכנן↔בפועל ברמת הקו עבור הקבצים הנוכחיים.

## שאלה 2 — באיזו תדירות פעלו קווי ה-rides בפועל ב-april/may?
מספר נסיעות לקו ליום (ממוצע), השוואה בין אפריל למאי.

In [3]:
apr_days = apr['date'].nunique()
may_days = may['date'].nunique()

freq = pd.DataFrame({
    'apr_rides': apr.groupby('line').size(),
    'may_rides': may.groupby('line').size(),
}).fillna(0).astype(int)
freq['apr_per_day'] = (freq['apr_rides'] / apr_days).round(1)
freq['may_per_day'] = (freq['may_rides'] / may_days).round(1)
freq['delta_per_day'] = (freq['may_per_day'] - freq['apr_per_day']).round(1)
freq = freq.sort_values('apr_per_day', ascending=False)
freq

,apr_rides,may_rides,apr_per_day,may_per_day,delta_per_day
line,,,,,
501,555,450,111.0,112.5,1.5
48,519,412,103.8,103.0,-0.8
47,505,404,101.0,101.0,0.0
24,418,349,83.6,87.2,3.6
249,407,341,81.4,85.2,3.8
90,378,373,75.6,93.2,17.6
561,378,274,75.6,68.5,-7.1
12,342,290,68.4,72.5,4.1
21,337,260,67.4,65.0,-2.4


In [4]:
print('סה"כ קווים בפועל:', freq.shape[0])
print('קווים בשני החודשים:', int(((freq.apr_rides>0)&(freq.may_rides>0)).sum()))
print('שינוי ממוצע נסיעות/יום (מאי מול אפריל):', round(freq['delta_per_day'].mean(), 1))
print('\nקווים שירדו הכי הרבה בתדירות:')
print(freq.nsmallest(3, 'delta_per_day')[['apr_per_day','may_per_day','delta_per_day']])

סה"כ קווים בפועל: 18
קווים בשני החודשים: 18
שינוי ממוצע נסיעות/יום (מאי מול אפריל): 0.3

קווים שירדו הכי הרבה בתדירות:
      apr_per_day  may_per_day  delta_per_day
line                                         
8            66.0         54.0          -12.0
6            42.4         33.8           -8.6
6א           31.2         23.0           -8.2


## שאלה 3 — האם זמני ההתחלה תואמים בין מתוכנן לבפועל?
מאחר שאין קווים משותפים, לא ניתן להשוות זמן-מתוכנן (`trips.departure_min`) מול זמן-בפועל (`rides.start_time`) באותו קו.
במקום זאת מוצגות שתי הפלגות הזמן בנפרד: שעות היציאה בפועל ב-rides, ושעות היציאה המתוכננות ב-trips.

In [5]:
# בפועל — התפלגות שעת יציאה (rides, אפריל)
print('שעת יציאה בפועל (rides אפריל) — התפלגות לפי שעה:')
print(apr['hour'].value_counts().sort_index().to_dict())

# טווח שעות פעילות לכל קו בפועל
span = apr.groupby('line')['hour'].agg(['min','max','count']).rename(
    columns={'min':'first_hour','max':'last_hour','count':'rides'})
span = span.sort_values('rides', ascending=False)
span

שעת יציאה בפועל (rides אפריל) — התפלגות לפי שעה:
{0: 48, 4: 5, 5: 221, 6: 346, 7: 379, 8: 361, 9: 333, 10: 332, 11: 333, 12: 349, 13: 338, 14: 341, 15: 326, 16: 301, 17: 276, 18: 274, 19: 247, 20: 224, 21: 189, 22: 165, 23: 101}


,first_hour,last_hour,rides
line,,,
501,5,23,555
48,5,22,519
47,0,23,505
24,0,23,418
249,5,23,407
90,5,23,378
561,0,23,378
12,5,22,342
21,5,23,337


In [6]:
# מתוכנן — התפלגות שעת יציאה (trips). departure_min = דקות מחצות; day_offset מטופל.
trips_dep = trips.copy()
trips_dep['dep_hour'] = ((trips_dep['departure_min'] + trips_dep['day_offset']*1440) // 60).astype(int) % 24
print('שעת יציאה מתוכננת (trips) — התפלגות לפי שעה:')
print(trips_dep['dep_hour'].value_counts().sort_index().to_dict())
print('\nטווח departure_min:', int(trips['departure_min'].min()), '-', int(trips['departure_min'].max()), 'דקות')
print('הערה: ב-trips יש ערכי departure_min רבים = 0 (זמן יציאה לא אוכלס), לכן ההתפלגות חלקית.')

שעת יציאה מתוכננת (trips) — התפלגות לפי שעה:
{0: 25, 1: 1, 4: 3, 5: 37, 6: 76, 7: 88, 8: 80, 9: 72, 10: 75, 11: 70, 12: 75, 13: 85, 14: 83, 15: 83, 16: 81, 17: 80, 18: 76, 19: 78, 20: 71, 21: 66, 22: 64, 23: 52}

טווח departure_min: 0 - 1380 דקות
הערה: ב-trips יש ערכי departure_min רבים = 0 (זמן יציאה לא אוכלס), לכן ההתפלגות חלקית.


## הקשר: אפיון עצמאי של trips (המתוכנן)
תמצית, מאחר שלא ניתן לחבר ל-rides.

In [7]:
print('קווים ייחודיים (catalog_number):', trips['catalog_number'].nunique())
print('low_demand_flag:', trips['low_demand_flag'].value_counts().to_dict())
print('daily_passengers ריק:', trips['daily_passengers'].isna().all(),
      '| weekly_passengers ריק:', trips['weekly_passengers'].isna().all())
print('\nמשך נסיעה (דק):')
print(trips['duration_min'].describe()[['min','50%','mean','max']].round(1).to_dict())
print('\nמרחק (ק"מ):')
print(trips['distance_km'].describe()[['min','50%','mean','max']].round(2).to_dict())
print('\n5 הקווים עם הכי הרבה נסיעות מתוכננות:')
print(trips['catalog_number'].value_counts().head(5).to_dict())

קווים ייחודיים (catalog_number): 48
low_demand_flag: {True: 1421}
daily_passengers ריק: True | weekly_passengers ריק: True

משך נסיעה (דק):
{'min': 10.0, '50%': 70.0, 'mean': 67.8, 'max': 140.0}

מרחק (ק"מ):
{'min': 4.0, '50%': 18.29, 'mean': 20.32, 'max': 68.36}

5 הקווים עם הכי הרבה נסיעות מתוכננות:
{14068: 53, 47006: 50, 24004: 45, 69005: 42, 23056: 42}


## דוח ממצאים (עברית)

### 1. חיבור trips ↔ rides — לא קיים
- **0 מק"טים משותפים** בין 48 הקווים שב-`trips.csv` ל-17 הקווים שב-`gtfs_rides`. שתי הקבוצות זרות לחלוטין.
- לפי מספר-קו (sign) חופפים רק `6` ו-`8`, אך עם מק"ט שונה — מספרי קו נפוצים שחוזרים באזורים שונים, כלומר התאמה מקרית ולא אותו קו.
- **משמעות:** הקבצים מתעדים אוכלוסיות קווים שונות. `trips` = 48 קווי ביקוש-נמוך (כנראה מועמדים לצמצום/שינוי), `rides` = 17 קווים פעילים אחרים (501, 524, 525, 47, 48, 90, 249, 561...). לא ניתן לאמת "מתוכנן מול בפועל" עבור אותם קווים מתוך הנתונים הנוכחיים.

### 2. תדירות בפועל (rides) — אפריל מול מאי
- כל 18 מספרי-הקו שב-rides פעלו בשני החודשים (אפריל: 5 ימים, מאי: 4 ימים).
- הקווים העמוסים ביותר: **501** (~111–112 נסיעות/יום), **48** (~104), **47** (~101), **24** (~84–87), **249** (~81–85), **90** (~76–93).
- הקו הדליל ביותר: **909** (~3–4 נסיעות/יום בלבד).
- מגמת מאי מול אפריל יציבה ברוב הקווים; ראו עמודת `delta_per_day` בטבלה (למשל קו 6 ירד ~9 נסיעות/יום).

### 3. זמני התחלה
- **בפועל (rides):** רוב הקווים פועלים ~05:00–23:00; חלק (24, 47, 561, 8) עם נסיעות גם בשעות 00–04.
- **מתוכנן (trips):** שדה `departure_min` אינו מאוכלס בעקביות (ערכי 0 רבים), ולכן חלון הזמן המתוכנן חלקי.
- בהיעדר קווים משותפים לא ניתן להשוות זמן-יציאה מתוכנן מול בפועל באותו קו.

### 4. איכות נתוני trips
- 1,421 נסיעות, 48 קווים, **כולן** `low_demand_flag=True`.
- `daily_passengers` / `weekly_passengers` / `line_id` / `custom_json` / `existing_flag` — **ריקים לחלוטין**. `vehicle_type_ids` מאוכלס בכ-48%.

### מסקנה / המשך מומלץ
1. כדי לחבר מתוכנן↔בפועל צריך ייצוא `gtfs_rides` **עבור 48 המק"טים שב-trips** (לא קבוצת הקווים הנוכחית), או לקבל את המק"טים התואמים לקווי ה-rides.
2. נתוני הנוסעים ב-trips ריקים — אם נדרש ניתוח ביקוש בפועל, יש למשוך אותו ממקור נפרד (Stride / siri).
3. הקובץ הנוכחי שימושי לאפיון עצמאי של תדירות בפועל בקווי rides — ראו טבלת `freq`.